# RJ45 insertion: contact state, guided search and port memory

Can the robot tell from its own signals (wrist F/T, commanded vs measured TCP, joint torques) whether the
plug went in, sits on the chamfer, or is next to the port on the panel face, and use that to get it in?
And does remembering the recurring part of the perception error make the next insertion first-try?

What the laptop runs showed (36 + 24 scenes, nominal jack; details in `notes/INSERTION_STATE.md`):
* in vs out is identifiable 100 ms after contact (balanced acc 0.965, leave-scenes-out; 0.978 at 150 ms);
* on the chamfer ramp the lateral force points to the hole (median 31 deg), on the face it does not (104 deg);
* with retries, a contact-predicted correction beats a spiral search at 2 mm perception error
  (0.84 vs 0.73 success, paired +0.11 [+0.04, +0.18]); a true one-shot correction reaches 1.00;
* the privileged expert's own failures are the gripper hitting cables plugged in the row above, not alignment.

This notebook scales that up and adds: a 100 Hz F/T stream and a 1-D CNN ensemble (GPU) vs the linear
models; a tight, more realistic jack (0.6 mm clearance, 1 mm chamfer); belief-based search and sliding
into the hole instead of retracting; and the port-memory stream (E3).

**Run**: VS Code with a Colab A100 runtime (or Colab in the browser), Run All. `PRESET="a100"` takes
about 3 h; the simulation runs on the VM's CPUs, only the CNN uses the GPU. Every stage writes shards and
skips finished work, so after a disconnect just Run All again. Without a GPU the `smoke` preset runs
(about 10 min on a laptop).

In [ ]:
# Setup cell, inlined into every notebook by notebooks/build.py.
# Locally: finds the repo. On a Colab runtime (also VS Code connected to Colab), where the code is not
# on the VM: clones ROBOGYM_REPO (default: the public repo), pip-installs it, sets MUJOCO_GL=egl and
# keeps data/ + runs/ on Drive when it is mounted.
import os, pathlib, subprocess, sys

DEFAULT_REPO = "https://github.com/carlo-scr/RoboGym.git"
ROBOGYM_REPO = os.environ.get("ROBOGYM_REPO") or DEFAULT_REPO
USE_DRIVE = os.environ.get("ROBOGYM_DRIVE", "1") == "1"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
    _icd = pathlib.Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
    if subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0 and not _icd.exists():
        _icd.parent.mkdir(parents=True, exist_ok=True)
        _icd.write_text('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')
    CODE = pathlib.Path("/content/RoboGym")
    DRIVE = pathlib.Path("/content/drive/MyDrive")
    if (CODE / ".git").exists():
        subprocess.run(["git", "-C", str(CODE), "pull", "-q"])
    if not (CODE / "robogym").exists():
        subprocess.run(["git", "clone", "-q", ROBOGYM_REPO, str(CODE)], check=True)
    if USE_DRIVE and not DRIVE.exists():
        try:
            from google.colab import drive

            drive.mount("/content/drive")
        except Exception as e:  # noqa: BLE001
            print("Drive not mounted:", e)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{CODE}[gym]"], check=True)
    if DRIVE.exists():
        for d in ("data", "runs"):
            tgt = DRIVE / "robogym_runs" / d
            tgt.mkdir(parents=True, exist_ok=True)
            link = CODE / d
            if not link.is_symlink():
                subprocess.run(["rm", "-rf", str(link)])
                link.symlink_to(tgt)
    os.chdir(CODE)
ROOT = pathlib.Path.cwd()
while not (ROOT / "robogym").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
if sys.platform.startswith("linux") and not os.environ.get("DISPLAY"):
    os.environ.setdefault("MUJOCO_GL", "egl")
print("ROOT:", ROOT, "| colab:", IN_COLAB)

## Experiment code
The modules live in `RoboGym/experiments/insertion_state/` on the laptop and are not on the public repo
yet, so they are inlined here. On Colab they are written to the clone; locally an existing file is never
overwritten (set `INSERTION_FORCE_WRITE=1` to force).

In [ ]:
EXP = pathlib.Path(ROOT) / "experiments" / "insertion_state"
EXP.mkdir(parents=True, exist_ok=True)
FORCE_WRITE = os.environ.get("INSERTION_FORCE_WRITE") == "1"


def write_module(name, src):
    p = EXP / name
    if p.exists() and not (IN_COLAB or FORCE_WRITE):
        if p.read_text() != src:
            print(f"{name}: the local file differs from the notebook copy, keeping the local file")
        return
    p.write_text(src)


if str(EXP) not in sys.path:
    sys.path.insert(0, str(EXP))

In [ ]:
write_module('common.py', r'''"""Shared pieces for the insertion contact-state experiments (see notes/INSERTION_STATE.md).

Everything is expressed in the *believed* port frame: x along the insertion axis (into the panel),
y / z lateral. The believed port is the true port shifted by a perception error `err_yz`; the robot
only knows the believed frame, its own kinematics (commanded and measured TCP), the wrist wrench and
the joint torques. Privileged labels (depth, lateral error, contacts) are logged next to them.
"""
from __future__ import annotations

import mujoco
import numpy as np

from robogym import geometry as G
from robogym.env import ACT_DIM, RoboGymEnv
from robogym.scene import box
from robogym.tasks.cable_plug import CablePlug

MAX_LEAD = 0.005  # force-limited push, as the expert (~servo stiffness x 5 mm)
CATS = ("plug_face", "plug_wedge", "plug_back", "plug_occ", "grip_occ", "grip_panel", "cable_panel")
HR_EVERY = 5  # wrench sample every 5 physics steps of 2 ms: 100 Hz (assumed F/T stream rate)
JACKS = {  # jack variants (task config overrides); "tight" is closer to a real RJ45 jack
    "nominal": {},
    "tight": {"task.clearance": 0.0006, "task.chamfer": 0.001},
}

# Sim fix (experiment-local): the face strips only cover +-rows*row_pitch/2 and the housing behind them is
# visual, so a plug that skids off the strips can pass *under* the panel and register as inserted.
# Close the face below the bottom row with a collision cap. Not above the top row: there the visual housing
# is what the top-down gripper body passes through when plugging the bottom row (a real clearance problem
# of the cell geometry, see notes/INSERTION_STATE.md), and a cap there stops bottom-row plugs 0.4 mm short.
_panel_xml_orig = CablePlug._panel_xml


def _panel_xml_solid(self):
    xml, assets = _panel_xml_orig(self)
    tc = self.tc
    t, hw = tc.panel_thickness, tc.panel_half_width
    face_h = tc.rows * tc.row_pitch / 2
    cap = ((tc.rows * tc.row_pitch + 0.02) / 2 - face_h) / 2
    return xml + "\n" + box("panel_cap_bot", (t / 2, 0, -(face_h + cap)), (t / 2, hw, cap), cls="port"), assets


CablePlug._panel_xml = _panel_xml_solid


def make_env(overrides=None):
    return RoboGymEnv("cable_plug", render=False, **(overrides or {}))


def prepare(env, seed, variant=None):
    """Run the expert until it starts the insertion; return (expert, snapshot) or None if it never gets there."""
    env.reset(seed=seed, variant=dict({"n": 1}, **(variant or {})))
    ex = env.task.make_expert(env, seed=seed)
    while not ex.done:
        a = ex.act()
        if ex.phase == "insert":
            return ex, env.get_state()
        env.step(a)
    return None


class Probe:
    """Geometry + contact bookkeeping for one env (geom-id sets are built once)."""

    def __init__(self, env, c=0):
        self.env, self.c = env, c
        m, task = env.model, env.task
        name = lambda g: m.geom(g).name or ""  # noqa: E731
        self.plug = set(task.plug_geoms[c])
        self.cable = {g for g in range(m.ngeom) if name(g).startswith(f"cable{c}_")}
        self.occ = {g for g in range(m.ngeom) if name(g).startswith("occ")}
        self.face = {g for g in task.panel_geoms if name(g).startswith("panel")}
        self.wedge = {g for g in task.panel_geoms if name(g).startswith("port") and not name(g).endswith("_back")}
        self.back = {g for g in task.panel_geoms if name(g).endswith("_back")}
        self.panel = set(task.panel_geoms)
        arm = env.arms[0]
        pre = arm[0] + "_"
        self.grip = {g for g in range(m.ngeom) if name(g).startswith(pre) and g not in env.arm[arm]["all_pads"]}
        self.pads = set(env.arm[arm]["all_pads"])
        self.arm = arm
        self.ft_site = env.arm[arm]["ft"]
        self._f6 = np.zeros(6)
        # high-rate wrench stream: sample inside env.step's physics loop
        self.hr, self._n_phys = [], 0
        physics_step = env._physics_step

        def logged(control=None):
            physics_step(control)
            self._n_phys += 1
            if self._n_phys % HR_EVERY == 0:
                self.hr.append(self.wrench_world())

        env._physics_step = logged

    def contact_forces(self):
        """Summed normal force per contact category."""
        m, d = self.env.model, self.env.data
        out = dict.fromkeys(CATS, 0.0)
        for i in range(d.ncon):
            con = d.contact[i]
            g1, g2 = con.geom1, con.geom2
            mujoco.mj_contactForce(m, d, i, self._f6)
            f = abs(self._f6[0])
            for a, b in ((g1, g2), (g2, g1)):
                if a in self.plug:
                    if b in self.face:
                        out["plug_face"] += f
                    elif b in self.wedge:
                        out["plug_wedge"] += f
                    elif b in self.back:
                        out["plug_back"] += f
                    elif b in self.occ:
                        out["plug_occ"] += f
                elif a in self.grip:
                    if b in self.occ:
                        out["grip_occ"] += f
                    elif b in self.panel:
                        out["grip_panel"] += f
                elif a in self.cable and (b in self.panel or b in self.occ):
                    out["cable_panel"] += f
        return out

    def wrench_world(self):
        w = self.env.wrench(self.arm)
        R = self.env.data.site_xmat[self.ft_site].reshape(3, 3)
        return np.concatenate([R @ w[:3], R @ w[3:]])

    def joint_torques(self):
        return self.env.data.actuator_force[self.env.arm[self.arm]["act"]].copy()


def true_port_frame(env, port):
    p, q = env.task.port_pose(env, port)
    return p, G.quat_to_mat(q)


class Attempt:
    """Non-privileged blind insertion from a snapshot, logging signals and labels at every control step.

    Phases: 'pre' (stiff: put the believed tip on the believed axis, `pre_x` in front of the believed
    face), 'push' (compliant: drive the tip target to seat depth + overshoot with a force-limited
    lead), 'hold'. Stops early on the latch click (observable on a real cable).
    """

    def __init__(self, env, probe, snap, port, err=(0.0, 0.0, 0.0), pre_x=-0.006, speed=0.02, overshoot=0.003, hold_s=0.5):
        self.env, self.pb, self.snap = env, probe, snap
        self.err = np.asarray(err, float)  # believed port - true port (port frame: axial, y, z)
        self.port = port
        self.pre_x, self.speed, self.overshoot, self.hold_s = pre_x, speed, overshoot, hold_s
        self.rows = []

    # -- frames
    def setup(self):
        env = self.env
        env.set_state(self.snap)
        p_port, R = true_port_frame(env, self.port)
        self.R = R  # orientation assumed known
        self.p_bel = p_port + R @ self.err
        pr = env.privileged()["plugs"][self.pb.c]
        self.tip0 = pr["tip"].copy()  # in-hand estimate (re-measured before insertion, like the expert)
        self.tgt0, self.q0 = env.pending_targets(self.pb.arm)
        self.grip = env.grip_cmd[self.pb.arm]
        self.tare = self.pb.wrench_world()
        self.tc = env.task.tc
        self.n_hr = env.n_sub // HR_EVERY
        self.pb.hr.clear()
        self.pb._n_phys = 0

    def to_bel(self, p):
        return self.R.T @ (np.asarray(p) - self.p_bel)

    def from_bel(self, x):
        return self.p_bel + self.R @ np.asarray(x)

    def tcp_target_for_tip(self, tip_bel):
        """TCP target that would put the tip at `tip_bel` (believed frame) if tracking were perfect."""
        return self.tgt0 + (self.from_bel(tip_bel) - self.tip0)

    def action(self, tp, compliant):
        env = self.env
        a = np.zeros(ACT_DIM * len(env.arms))
        p_cur, q_cur = env.pending_targets(self.pb.arm)
        a[:3] = tp - p_cur
        a[3:6] = G.quat_error(self.q0, q_cur)
        a[6] = self.grip
        a[7] = float(compliant)
        return a

    def step(self, tp, compliant, phase):
        self.env.step(self.action(tp, compliant))
        self.log(phase, tp)

    def log(self, phase, tp_cmd):
        env, pb = self.env, self.pb
        pr = env.privileged()["plugs"][pb.c]
        p_tcp, _ = env.tcp_pose(pb.arm)
        w = pb.wrench_world() - self.tare
        k = pr["nearest_port"]
        p_port, R = true_port_frame(env, self.port)
        rel = R.T @ (pr["tip"] - p_port)
        hr = np.array(pb.hr[-self.n_hr :]) if pb.hr else np.tile(w + self.tare, (self.n_hr, 1))
        hr = np.concatenate([np.repeat(hr[:1], self.n_hr - len(hr), 0), hr]) if len(hr) < self.n_hr else hr
        hr = hr - self.tare
        pb.hr.clear()
        self.rows.append(dict(
            t=env.time, phase=phase,
            cmd=self.to_bel(tp_cmd + (self.tip0 - self.tgt0)),  # commanded tip (believed frame)
            tcp=self.to_bel(p_tcp + (self.tip0 - self.tgt0)),  # measured TCP mapped to the tip (believed frame)
            F=self.R.T @ w[:3], M=self.R.T @ w[3:], tau=pb.joint_torques(),
            Fh=np.concatenate([hr[:, :3] @ self.R, hr[:, 3:] @ self.R], 1),  # 100 Hz wrench in the believed frame
            # privileged labels
            tip_true=rel, port_k=k, on_target=k == self.port, angle=pr["angle"], latched=pr["latched"],
            grasped=pr["grasped_by"] is not None, **{f"f_{c}": v for c, v in pb.contact_forces().items()},
        ))

    def run(self):
        self.setup()
        env, dt = self.env, self.env.dt
        # pre: stiff lateral correction onto the believed axis, pre_x in front of the believed face
        start = self.to_bel(self.tip0)
        goal = np.array([self.pre_x, 0.0, 0.0])
        n = max(4, int(round(0.6 / dt)))
        for i in range(1, n + 1):
            u = G.min_jerk(i / n)
            self.step(self.tcp_target_for_tip(start + u * (goal - start)), False, "pre")
        # push: compliant, force-limited, to seat depth + overshoot
        x1 = self.tc.seat_depth + self.overshoot
        n = max(1, int(round((x1 - self.pre_x) / self.speed / dt)))
        nh = int(round(self.hold_s / dt))
        for i in range(1, n + nh + 1):
            u = G.min_jerk(min(1.0, i / n))
            tp = self.tcp_target_for_tip(goal + u * np.array([x1 - self.pre_x, 0, 0]))
            p_tcp, _ = env.tcp_pose(self.pb.arm)
            lead = (tp - p_tcp) @ self.R[:, 0]
            if lead > MAX_LEAD:
                tp = tp - (lead - MAX_LEAD) * self.R[:, 0]
            self.step(tp, True, "push" if i <= n else "hold")
            if env.task.latched[self.pb.c] >= 0:
                break
        return self.rows


def outcome(rows, tc):
    """Privileged end-state class of an attempt. The port site is on the face plane (depth 0); the chamfer
    occupies depths (0, tc.chamfer]."""
    r = rows[-1]
    if r["latched"] >= 0:
        return "seated" if r["on_target"] else "wrong_port"
    if not r["grasped"]:
        return "dropped"
    depth = r["tip_true"][0]
    if r["on_target"] and depth > tc.chamfer + 0.0005:
        return "in_channel"
    if r["on_target"] and depth > 0.0003:
        return "chamfer"
    blocked = max(x["f_grip_occ"] + x["f_grip_panel"] + x["f_plug_occ"] for x in rows[-5:])
    return "blocked" if blocked > 5.0 else "face"


def stack(rows, key):
    return np.array([r[key] for r in rows])
''')

In [ ]:
write_module('e1_offset_sweep.py', r'''"""E1: blind compliant insertions from the pre-insertion snapshot with controlled port-estimate errors.
Lateral error on a polar grid, axial error uniform in +-2 mm. Saves every attempt's time series, one
shard per scene (`<out>/scene_<seed>.pkl`); rerunning skips scenes that already have a shard."""
from __future__ import annotations

import argparse
import json
import pickle
import sys
from multiprocessing import Pool
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).parent))
from common import JACKS, Attempt, Probe, make_env, outcome, prepare  # noqa: E402

OUT = Path(__file__).resolve().parents[2] / "runs" / "insertion_state"
RADII_MM = (0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0)
N_ANG = 12
KEYS_VEC = ("cmd", "tcp", "F", "M", "tau", "tip_true", "Fh")
KEYS_SCALAR = ("t", "angle", "latched", "on_target", "f_plug_face", "f_plug_wedge", "f_plug_back", "f_plug_occ", "f_grip_occ", "f_grip_panel", "f_cable_panel")
PHASES = ("pre", "push", "hold", "deeper", "move", "retract", "reaim", "slide")  # E1 uses the first three


def offsets(seed, radii=RADII_MM):
    rng = np.random.default_rng([seed, 11])
    out = [(0.0, 0.0)]
    for r in radii:
        a0 = rng.uniform(0, 2 * np.pi / N_ANG)
        out += [(r * np.cos(a0 + 2 * np.pi * i / N_ANG), r * np.sin(a0 + 2 * np.pi * i / N_ANG)) for i in range(N_ANG)]
    ax = rng.uniform(-2.0, 2.0, len(out))
    return [np.array([x, y, z]) * 1e-3 for x, (y, z) in zip(ax, out)]


def pack(rows):
    d = {k: np.array([r[k] for r in rows], np.float32) for k in KEYS_VEC + KEYS_SCALAR}
    d["phase"] = np.array([PHASES.index(r["phase"]) for r in rows], np.int8)
    return d


def run_scene(job):
    seed, jack, out_dir, radii = job
    shard = Path(out_dir) / f"scene_{seed}.pkl"
    if shard.exists():
        return seed
    env = make_env(JACKS[jack])
    got = prepare(env, seed)
    res = []
    if got is not None:
        _ex, snap = got
        port = int(env.scenario.task["cables"][0]["port"])
        pb = Probe(env)
        for err in offsets(seed, radii):
            rows = Attempt(env, pb, snap, port, err=err).run()
            res.append(dict(scene=seed, port=port, jack=jack, err=err, outcome=outcome(rows, env.task.tc), unstable=bool(env.unstable), ts=pack(rows)))
            env.unstable = False
    tmp = shard.with_suffix(".tmp")
    with open(tmp, "wb") as f:
        pickle.dump(res, f)
    tmp.rename(shard)
    return seed


def load(path):
    """All attempts from a shard directory (or a single pickle from older runs)."""
    path = Path(path)
    if path.is_dir():
        res = []
        for f in sorted(path.glob("scene_*.pkl")):
            res += pickle.load(open(f, "rb"))
        return res
    return pickle.load(open(path, "rb"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--scenes", type=int, nargs="+", default=None)
    ap.add_argument("--n", type=int, default=36)
    ap.add_argument("--seed0", type=int, default=1000)
    ap.add_argument("--jack", default="nominal", choices=sorted(JACKS))
    ap.add_argument("--radii", default=None, help="json list of radii in mm (default: the full grid)")
    ap.add_argument("--workers", type=int, default=12)
    ap.add_argument("--name", default="e1")
    ap.add_argument("--out", default=None, help="shard directory (default runs/insertion_state/<name>)")
    args = ap.parse_args()
    scenes = args.scenes or list(range(args.seed0, args.seed0 + args.n))
    out_dir = Path(args.out) if args.out else OUT / args.name
    out_dir.mkdir(parents=True, exist_ok=True)
    radii = tuple(json.loads(args.radii)) if args.radii else RADII_MM
    jobs = [(s, args.jack, str(out_dir), radii) for s in scenes]
    with Pool(args.workers) as p:
        for i, s in enumerate(p.imap_unordered(run_scene, jobs), 1):
            print(f"  scene {s} done ({i}/{len(jobs)})", flush=True)
    res = [a for a in load(out_dir) if a["scene"] in set(scenes)]
    oc = {}
    for a in res:
        oc[a["outcome"]] = oc.get(a["outcome"], 0) + 1
    print(f"{len(res)} attempts over {len({a['scene'] for a in res})} scenes; unstable {sum(a['unstable'] for a in res)}; outcomes {oc}")
    r = np.array([np.linalg.norm(a["err"][1:]) * 1e3 for a in res])
    seated = np.array([a["outcome"] == "seated" for a in res])
    for rr in (0.0,) + radii:
        sel = np.isclose(r, rr, atol=1e-3)
        print(f"  r={rr:4.1f} mm  seated {seated[sel].mean():.2f}  " + " ".join(f"{k}={np.mean([res[i]['outcome'] == k for i in np.where(sel)[0]]):.2f}" for k in ("in_channel", "chamfer", "face", "blocked")))


if __name__ == "__main__":
    main()
''')

In [ ]:
write_module('analyze_e1.py', r'''"""E1 analysis: is the contact state identifiable from non-privileged signals shortly after first contact,
and does the contact tell the robot which way to correct? Leave-scenes-out evaluation.

Contact is detected without privileged information (|F| > F_ON after taring, with sensor noise); the
labels come from the simulator at the same moment:
  in       the tip passes the face plane within 3 steps (entered the channel, possibly via the chamfer)
  ramp     the plug edge overlaps the jack wall by <= the chamfer width (on the lead-in, should be guidable)
  face     overlap beyond the chamfer (wedge front or face strip: no lateral guidance from the geometry)
  blocked  the gripper / plug touches a neighbouring cable first
"""
from __future__ import annotations

import argparse
import pickle
from pathlib import Path

import numpy as np

OUT = Path(__file__).resolve().parents[2] / "runs" / "insertion_state"
F_ON = 2.5  # N, contact detection threshold on the tared wrench
SIG_F, SIG_M, SIG_TAU = 0.3, 0.01, 0.2  # assumed sensor noise (N, Nm, Nm)
HALF_CLEAR = 0.0008  # (channel - plug) / 2 per side
CHAMFER = 0.0025
CLASSES = ("in", "ramp", "face", "blocked")
K_MAX = 3


def noisy(a, i):
    rng = np.random.default_rng([i, 5])
    ts = a["ts"]
    F = ts["F"] + rng.normal(0, SIG_F, ts["F"].shape)
    M = ts["M"] + rng.normal(0, SIG_M, ts["M"].shape)
    tau = ts["tau"] + rng.normal(0, SIG_TAU, ts["tau"].shape)
    return F, M, tau


def first_contact(F, phase):
    idx = np.where((phase >= 1) & (np.linalg.norm(F, axis=1) > F_ON))[0]
    return int(idx[0]) if len(idx) else None


def label(ts, c0):
    """Privileged contact-state label at the (non-privileged) detection step c0."""
    w = slice(max(0, c0 - 1), c0 + 2)
    blocked = (ts["f_grip_occ"][w] + ts["f_grip_panel"][w] + ts["f_plug_occ"][w]).max()
    plug = (ts["f_plug_face"][w] + ts["f_plug_wedge"][w] + ts["f_plug_back"][w]).max()
    if blocked > 1.0 and blocked >= plug:
        return "blocked"
    depth = ts["tip_true"][c0 : c0 + 4, 0]
    if (depth > 0.0003).any() or (ts["latched"][c0 : c0 + 4] >= 0).any():
        return "in"
    y, z = np.abs(ts["tip_true"][c0, 1:])
    overlap = max(y, z) - HALF_CLEAR  # crude: the larger-axis overlap of the plug edge into the jack wall
    return "ramp" if overlap <= CHAMFER else "face"


def features(F, M, tau, ts, c0, k):
    """Non-privileged features at step c0 + k relative to the last pre-contact step c0 - 1."""
    b = max(0, c0 - 1)
    j = min(c0 + k, len(F) - 1)
    lag = ts["tcp"] - ts["cmd"]
    rows = []
    for s in range(c0, j + 1):
        rows.append(np.concatenate([
            F[s], M[s],
            [lag[s, 0] - lag[b, 0], ts["tcp"][s, 0] - ts["tcp"][b, 0]],  # axial lag growth, axial travel since contact
            lag[s, 1:] - lag[b, 1:],  # lateral deflection since contact
            tau[s] - tau[b],
        ]))
    while len(rows) < k + 1:
        rows.append(rows[-1])
    return np.concatenate(rows).astype(np.float64)


FEAT_NAMES_STEP = ["Fx", "Fy", "Fz", "Mx", "My", "Mz", "lag_x", "travel_x", "defl_y", "defl_z"] + [f"tau{i}" for i in range(7)]


def build(res, k):
    X, y, corr, scene, aux = [], [], [], [], []
    for i, a in enumerate(res):
        ts = a["ts"]
        F, M, tau = noisy(a, i)
        c0 = first_contact(F, ts["phase"])
        if c0 is None:
            continue
        X.append(features(F, M, tau, ts, c0, k))
        y.append(CLASSES.index(label(ts, c0)))
        corr.append(-ts["tip_true"][c0, 1:])  # lateral move that would centre the tip on the port
        scene.append(a["scene"])
        aux.append(dict(i=i, c0=c0, seated=bool(ts["latched"][-1] >= 0), err=a["err"]))
    return np.array(X), np.array(y), np.array(corr), np.array(scene), aux


# ---------------------------------------------------------------- small numpy models
class Std:
    def fit(self, X):
        self.mu, self.sd = X.mean(0), X.std(0) + 1e-9
        return self

    def __call__(self, X):
        return (X - self.mu) / self.sd


def fit_logreg(X, y, n_cls, lam=1e-2, iters=600, lr=0.5):
    n, d = X.shape
    W = np.zeros((d + 1, n_cls))
    Xb = np.hstack([X, np.ones((n, 1))])
    Y = np.eye(n_cls)[y]
    w_cls = 1.0 / np.maximum(1, np.bincount(y, minlength=n_cls))  # class-balanced
    sw = w_cls[y] / w_cls[y].mean()
    for _ in range(iters):
        Z = Xb @ W
        Z -= Z.max(1, keepdims=True)
        P = np.exp(Z)
        P /= P.sum(1, keepdims=True)
        G = Xb.T @ ((P - Y) * sw[:, None]) / n + lam * np.vstack([W[:-1], np.zeros((1, n_cls))])
        W -= lr * G
    return W


def predict_logreg(W, X):
    return np.argmax(np.hstack([X, np.ones((len(X), 1))]) @ W, 1)


def fit_ridge(X, Y, lam=1.0):
    Xb = np.hstack([X, np.ones((len(X), 1))])
    return np.linalg.solve(Xb.T @ Xb + lam * np.eye(Xb.shape[1]), Xb.T @ Y)


def predict_ridge(B, X):
    return np.hstack([X, np.ones((len(X), 1))]) @ B


def balanced_acc(y, p, n_cls):
    accs = [np.mean(p[y == c] == c) for c in range(n_cls) if (y == c).any()]
    return float(np.mean(accs))


def ang_err(pred, true):
    a = np.arctan2(pred[:, 1], pred[:, 0]) - np.arctan2(true[:, 1], true[:, 0])
    return np.degrees(np.abs((a + np.pi) % (2 * np.pi) - np.pi))


def folds(scene, n_folds=6, seed=0):
    us = np.unique(scene)
    rng = np.random.default_rng(seed)
    rng.shuffle(us)
    return [np.isin(scene, us[f::n_folds]) for f in range(n_folds)]


def cv_eval(X, y, corr, scene):
    n_cls = len(CLASSES)
    p_multi = np.zeros(len(y), int)
    p_bin = np.zeros(len(y), int)
    p_rule = np.zeros(len(y), int)
    p_corr = np.zeros_like(corr)
    p_corr_rule = np.zeros_like(corr)
    i_travel = FEAT_NAMES_STEP.index("travel_x")
    n_step = len(FEAT_NAMES_STEP)
    for te in folds(scene):
        tr = ~te
        sd = Std().fit(X[tr])
        W = fit_logreg(sd(X[tr]), y[tr], n_cls)
        p_multi[te] = predict_logreg(W, sd(X[te]))
        yb = (y != 0).astype(int)
        Wb = fit_logreg(sd(X[tr]), yb[tr], 2)
        p_bin[te] = predict_logreg(Wb, sd(X[te]))
        # one-feature rule: axial travel since contact at the last step, best threshold on train
        trav = X[:, (X.shape[1] // n_step - 1) * n_step + i_travel]
        ths = np.quantile(trav[tr], np.linspace(0.01, 0.99, 99))
        best = max(ths, key=lambda th: balanced_acc(yb[tr], (trav[tr] < th).astype(int), 2))
        p_rule[te] = (trav[te] < best).astype(int)
        # direction: ridge on all features, trained on out-of-jack plug contacts (ramp + face)
        m_tr = tr & np.isin(y, [1, 2])
        B = fit_ridge(sd(X[m_tr]), corr[m_tr], lam=10.0)
        p_corr[te] = predict_ridge(B, sd(X[te]))
        # rule: correction along +/- the lateral force at contact, sign fit on train per true class
        # (an upper bound for the rule: it is told whether the contact is ramp or face)
        Flat = X[:, 1:3]
        for cls in (1, 2):
            m_c = tr & (y == cls)
            s = np.sign(np.sum(np.einsum("ij,ij->i", Flat[m_c], corr[m_c]))) if m_c.any() else 1.0
            p_corr_rule[te & (y == cls)] = s * Flat[te & (y == cls)]
    return p_multi, p_bin, p_rule, p_corr, p_corr_rule


def report(k, X, y, corr, scene, aux, preds, log=print):
    p_multi, p_bin, p_rule, p_corr, p_corr_rule = preds
    yb = (y != 0).astype(int)
    out = dict(k=k, ms=50 * (k + 1), n=len(y), inout_logreg=balanced_acc(yb, p_bin, 2), inout_travel_rule=balanced_acc(yb, p_rule, 2),
               cls4_logreg=balanced_acc(y, p_multi, 4))
    log(f"\n--- {50 * (k + 1)} ms after contact (k={k}); n={len(y)} attempts, {len(np.unique(scene))} scenes; class counts "
        + ", ".join(f"{c}={int((y == i).sum())}" for i, c in enumerate(CLASSES)))
    log(f"  in vs out   : logistic balanced acc {out['inout_logreg']:.3f}   travel-threshold rule {out['inout_travel_rule']:.3f}")
    log(f"  4-class     : logistic balanced acc {out['cls4_logreg']:.3f}")
    cm = np.zeros((4, 4), int)
    for t, p in zip(y, p_multi):
        cm[t, p] += 1
    log("  confusion (rows true, cols pred): " + "  ".join(f"{c[:4]}" for c in CLASSES))
    for i, c in enumerate(CLASSES):
        log(f"     {c:8s} {cm[i]}")
    for name, cls in (("ramp", 1), ("face", 2)):
        m = y == cls
        if m.sum() < 5:
            continue
        e_l = ang_err(p_corr[m], corr[m])
        e_r = ang_err(p_corr_rule[m], corr[m])
        mag = np.linalg.norm(corr[m], axis=1) * 1e3
        mag_err = np.abs(np.linalg.norm(p_corr[m], axis=1) * 1e3 - mag)
        out[f"dir_{name}_ridge"], out[f"dir_{name}_force_rule"] = float(np.median(e_l)), float(np.median(e_r))
        log(f"  direction on {name:4s} (n={m.sum():4d}, true miss {np.median(mag):.1f} mm): learned median {np.median(e_l):5.1f} deg "
            f"(<45 deg: {np.mean(e_l < 45):.2f}), |mag| err {np.median(mag_err):.2f} mm;  lateral-force rule median {np.median(e_r):5.1f} deg")
    return out


def figures(res, X, y, corr, aux, preds_k, fig_dir=None):
    import matplotlib

    if fig_dir is None:
        matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig_dir = Path(fig_dir) if fig_dir else OUT / "figs"
    fig_dir.mkdir(parents=True, exist_ok=True)
    cols = {"in": "#2a9d8f", "ramp": "#e9c46a", "face": "#e76f51", "blocked": "#6c757d"}
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.6))
    # (a) capture map in true-miss coordinates at contact
    for i, c in enumerate(CLASSES):
        m = y == i
        ax[0].scatter(-corr[m, 0] * 1e3, -corr[m, 1] * 1e3, s=5, c=cols[c], label=c, alpha=0.6)
    for r in (HALF_CLEAR, HALF_CLEAR + CHAMFER):
        ax[0].add_patch(plt.Rectangle((-r * 1e3, -r * 1e3), 2 * r * 1e3, 2 * r * 1e3, fill=False, ls="--", lw=0.8, color="k"))
    ax[0].set(xlabel="tip y - port y at contact [mm]", ylabel="tip z - port z [mm]", title="contact state vs true miss", xlim=(-9, 9), ylim=(-9, 9), aspect="equal")
    ax[0].legend(fontsize=8, markerscale=3)
    # (b) axial force / travel after contact
    for i, c in enumerate(CLASSES[:3]):
        idx = np.where(y == i)[0][:150]
        for j in idx:
            a = res[aux[j]["i"]]
            ts, c0 = a["ts"], aux[j]["c0"]
            s = slice(max(0, c0 - 2), min(len(ts["t"]), c0 + 8))
            tt = (np.arange(s.start, s.stop) - c0) * 50
            ax[1].plot(tt, (ts["tcp"][s, 0] - ts["tcp"][max(0, c0 - 1), 0]) * 1e3, c=cols[c], lw=0.5, alpha=0.3)
            ax[2].plot(tt, ts["F"][s, 0], c=cols[c], lw=0.5, alpha=0.3)
    ax[1].set(xlabel="ms since detected contact", ylabel="axial TCP travel since contact [mm]", title="measured TCP (encoders)")
    ax[2].set(xlabel="ms since detected contact", ylabel="axial wrist force [N]", title="wrist F/T")
    for c in CLASSES[:3]:
        ax[1].plot([], [], c=cols[c], label=c)
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(fig_dir / "e1_contact_state.png", dpi=130)
    # (c) direction predictions on ramp / face
    p_corr = preds_k[3]
    fig, ax = plt.subplots(1, 2, figsize=(9, 4.4))
    for a_, cls, nm in ((ax[0], 1, "ramp"), (ax[1], 2, "face")):
        m = y == cls
        e = ang_err(p_corr[m], corr[m])
        a_.hist(e, bins=36, range=(0, 180), color=cols[nm])
        a_.set(title=f"{nm}: correction-direction error (learned)", xlabel="deg", ylabel="attempts")
    fig.tight_layout()
    fig.savefig(fig_dir / "e1_direction.png", dpi=130)
    print(f"\nfigures -> {fig_dir}")


def load_runs(name_or_path):
    """E1 attempts from a shard directory, a pickle, or a run name under runs/insertion_state/."""
    p = Path(name_or_path)
    if not p.exists():
        p = OUT / name_or_path if (OUT / name_or_path).is_dir() else OUT / f"{name_or_path}.pkl"
    if p.is_dir():
        res = []
        for f in sorted(p.glob("scene_*.pkl")):
            res += pickle.load(open(f, "rb"))
    else:
        res = pickle.load(open(p, "rb"))
    return [a for a in res if not a["unstable"]]


def analyze(res, ks=range(K_MAX + 1), fig_dir=None, log=print):
    log(f"{len(res)} attempts; end outcomes: " + str({o: sum(a['outcome'] == o for a in res) for o in sorted({a['outcome'] for a in res})}))
    rows, keep = [], None
    for k in ks:
        X, y, corr, scene, aux = build(res, k)
        preds = cv_eval(X, y, corr, scene)
        rows.append(report(k, X, y, corr, scene, aux, preds, log=log))
        if k == 1:
            keep = (X, y, corr, aux, preds)
    if keep is not None:
        figures(res, *keep, fig_dir=fig_dir)
    return rows


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--name", default="e1")
    args = ap.parse_args()
    analyze(load_runs(args.name))


if __name__ == "__main__":
    main()
''')

In [ ]:
write_module('e2_guided.py', r'''"""E2: does the contact state guide the plug in? Retrying insertion controllers that share one in/out
detector (logistic model from E1, 100 ms after contact) and the same "partial -> push deeper" rule, and
differ in what they do after a miss:
  blind    one push, no retry (E1's controller)
  spiral   sunflower search around the first aim (r_n = 1 mm * sqrt(n), golden angle)
  learned  shift the aim by the correction predicted from the contact signals (ridge model from E1)
  belief   grid belief over the hole offset (negative information + predicted correction, belief.py);
           aim where the posterior probability of entering is highest (also for the first try)
  slide    belief, but instead of retracting after a miss, slide to the new aim under a light axial
           preload and let the plug drop in; retract only if it does not
  oracle   shift the aim by the true lateral miss at contact (privileged upper bound for one-shot correction)
Perception error: lateral N(0, sigma) per axis, axial U(-2, 2) mm. New scenes (not in E1).
Results are written per scene (`<out>/scene_<seed>.json`); rerunning skips finished scenes."""
from __future__ import annotations

import argparse
import json
import sys
from multiprocessing import Pool
from pathlib import Path

import numpy as np

from robogym import geometry as G

sys.path.insert(0, str(Path(__file__).parent))
import analyze_e1 as A  # noqa: E402
from belief import CaptureMap, HoleBelief  # noqa: E402
from common import JACKS, MAX_LEAD, Attempt, Probe, make_env, outcome, prepare  # noqa: E402
from e1_offset_sweep import pack  # noqa: E402

OUT = A.OUT
K = 1  # decide 100 ms after contact (steps c0, c0 + 1)
GOLDEN = np.pi * (3 - np.sqrt(5))
STRATS = ("blind", "spiral", "learned", "belief", "slide", "oracle")
SLIDE_LEAD = 0.0015  # axial preload while sliding (force-limited lead, stiff lateral tracking)
SLIDE_SPEED = 0.010  # m/s
DROP = 0.0015  # axial advance beyond the contact position that counts as "dropped in"


def train_models(src="e1"):
    """Linear in/out + 4-class models, ridge correction with cross-validated residual covariances per
    class, and the capture map P(entry | tip offset at contact), all from E1 attempts."""
    res = A.load_runs(src)
    X, y, corr, scene, _aux = A.build(res, K)
    sd = A.Std().fit(X)
    W_bin = A.fit_logreg(sd(X), (y != 0).astype(int), 2)
    W4 = A.fit_logreg(sd(X), y, 4)
    m = np.isin(y, [1, 2])
    B = A.fit_ridge(sd(X[m]), corr[m], lam=10.0)
    resid = np.zeros_like(corr)
    for te in A.folds(scene):
        mt = ~te & m
        Bf = A.fit_ridge(A.Std().fit(X[~te])(X[mt]), corr[mt], lam=10.0)
        resid[te] = A.predict_ridge(Bf, A.Std().fit(X[~te])(X[te])) - corr[te]
    S = {c: np.cov(resid[y == i].T) + np.eye(2) * 1e-8 for c, i in (("ramp", 1), ("face", 2))}
    keep = y != 3  # the capture map is about plug contacts, not blocked grippers
    cap = CaptureMap(-corr[keep], y[keep] == 0)
    # systematic pre-contact drift of the compliant controller (tip offset at contact minus the aim, error
    # removed): every strategy compensates it in its first aim, as a calibrated controller would
    drift = np.array([-c - a["err"][1:] for c, a in zip(corr, _aux)]).mean(0)
    return dict(mu=sd.mu, sd=sd.sd, W_bin=W_bin, W4=W4, B=B, S_ramp=S["ramp"], S_face=S["face"], cap_P=cap.P, cap_half=cap.half, cap_res=cap.res,
                drift_mean=drift)


def capture_map(models):
    cap = CaptureMap.__new__(CaptureMap)
    cap.P, cap.half, cap.res = models["cap_P"], models["cap_half"], models["cap_res"]
    cap.n = cap.P.shape[0]
    return cap


def softmax(z):
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)


class Guided(Attempt):
    def __init__(self, *a, strategy="belief", models=None, max_tries=6, budget_s=9.0, noise_seed=0, prior_mean=(0.0, 0.0), prior_sigma=None,
                 **kw):
        super().__init__(*a, **kw)
        self.strategy, self.models = strategy, models
        self.max_tries = 1 if strategy == "blind" else max_tries
        self.budget_s = budget_s
        self.rng = np.random.default_rng([noise_seed, 17])
        self.prior_mean = np.asarray(prior_mean, float)
        self.prior_sigma = prior_sigma
        self.Fn, self.Mn, self.taun = [], [], []

    # noisy sensor stream, kept in step with self.rows
    def log(self, phase, tp_cmd):
        super().log(phase, tp_cmd)
        r = self.rows[-1]
        self.Fn.append(r["F"] + self.rng.normal(0, A.SIG_F, 3))
        self.Mn.append(r["M"] + self.rng.normal(0, A.SIG_M, 3))
        self.taun.append(r["tau"] + self.rng.normal(0, A.SIG_TAU, 7))

    def stiff_to(self, goal, T=0.4, phase="move"):
        start = self.rows[-1]["cmd"] if self.rows else self.to_bel(self.tip0)
        n = max(2, int(round(T / self.env.dt)))
        for i in range(1, n + 1):
            u = G.min_jerk(i / n)
            self.step(self.tcp_target_for_tip(start + u * (np.asarray(goal) - start)), False, phase)

    def decide(self, c0):
        ts = pack(self.rows)
        F, M, tau = np.array(self.Fn), np.array(self.Mn), np.array(self.taun)
        x = A.features(F, M, tau, ts, c0, K)[None]
        m = self.models
        xs = (x - m["mu"]) / m["sd"]
        out = A.predict_logreg(m["W_bin"], xs)[0] == 1
        p4 = softmax(np.hstack([xs, np.ones((1, 1))]) @ m["W4"])[0]
        p_ramp = p4[1] / max(p4[1] + p4[2], 1e-9)
        corr = A.predict_ridge(m["B"], xs)[0]
        true_corr = -self.rows[c0]["tip_true"][1:]
        return out, corr, true_corr, p_ramp

    def push(self, yz, x_to, phase="push", decide=True):
        """Compliant force-limited push along the believed axis at lateral aim yz. Returns a dict with
        res in ('latched' | 'out' | 'in_no_click') and, for 'out', the correction estimate, the true
        correction, p(ramp) and the contact step. decide=False (pushing deeper after an entry) skips the
        in/out check: the plug is already in contact."""
        env, dt = self.env, self.env.dt
        x0 = self.rows[-1]["cmd"][0]
        n = max(1, int(round((x_to - x0) / self.speed / dt)))
        nh = int(round(self.hold_s / dt))
        c0, verdict = None, None
        start = len(self.rows)
        for i in range(1, n + nh + 1):
            u = G.min_jerk(min(1.0, i / n))
            tp = self.tcp_target_for_tip(np.array([x0 + u * (x_to - x0), *yz]))
            p_tcp, _ = env.tcp_pose(self.pb.arm)
            lead = (tp - p_tcp) @ self.R[:, 0]
            if lead > MAX_LEAD:
                tp = tp - (lead - MAX_LEAD) * self.R[:, 0]
            self.step(tp, True, phase)
            if env.task.latched[self.pb.c] >= 0:
                return dict(res="latched")
            j = len(self.rows) - 1
            if decide and c0 is None and np.linalg.norm(self.Fn[j]) > A.F_ON:
                c0 = j
                self.drift_pre = (self.rows[max(start, j - 1)]["tcp"] - self.rows[max(start, j - 1)]["cmd"])[1:]
            if c0 is not None and verdict is None and j >= c0 + K:
                out, corr, true_corr, p_ramp = self.decide(c0)
                verdict = (corr, true_corr, p_ramp)
                if out:
                    return dict(res="out", corr=corr, true_corr=true_corr, p_ramp=p_ramp, c0=c0)
            if env.time - self.t_start > self.budget_s:
                break
        if not decide:
            return dict(res="in_no_click")
        if verdict is None:  # never touched hard enough to decide
            if self.rows[-1]["tcp"][0] > 0.004:
                return dict(res="in_no_click")
            return dict(res="out", corr=np.zeros(2), true_corr=-self.rows[-1]["tip_true"][1:], p_ramp=0.5, c0=len(self.rows) - 1)
        return dict(res="in_no_click")

    def slide(self, a_from, a_to, x_contact):
        """Stiff lateral slide from a_from to a_to with a light axial preload; 'dropped' when the measured
        tip advances DROP beyond the contact position, 'latched', or 'no_drop'."""
        env, dt = self.env, self.env.dt
        n = max(3, int(round(np.linalg.norm(np.asarray(a_to) - a_from) / SLIDE_SPEED / dt)))
        for i in range(1, n + int(0.2 / dt) + 1):
            u = G.min_jerk(min(1.0, i / n))
            lat = np.asarray(a_from) + u * (np.asarray(a_to) - a_from)
            x_meas = self.rows[-1]["tcp"][0]
            self.step(self.tcp_target_for_tip(np.array([x_meas + SLIDE_LEAD, *lat])), False, "slide")
            if env.task.latched[self.pb.c] >= 0:
                return "latched"
            if self.rows[-1]["tcp"][0] - x_contact > DROP:
                return "dropped"
            if env.time - self.t_start > self.budget_s:
                break
        return "no_drop"

    def run(self):
        self.setup()
        self.t_start = self.env.time
        self.drift_pre = np.zeros(2)
        self.drops = 0
        m = self.models
        belief = None
        u_mean = self.prior_mean + m["drift_mean"]  # prior over u = perception error + controller drift
        if self.strategy in ("belief", "slide"):
            sig = self.prior_sigma if self.prior_sigma is not None else self.sigma_default
            belief = HoleBelief(capture_map(m), sig, m["S_ramp"], m["S_face"], mean=u_mean)
            aim, _ = belief.best_aim()
        else:
            aim = -u_mean
        aim0 = aim.copy()
        x1 = self.tc.seat_depth + self.overshoot
        self.stiff_to([self.pre_x, *aim], T=0.6, phase="pre")
        tries, deeper, n_spiral, slid = 0, 0, 0, 0
        self.result = dict(tries=0, deeper=0, slides=0, success=False)
        while tries < self.max_tries and self.env.time - self.t_start < self.budget_s:
            tries += 1
            r = self.push(aim, x1)
            if r["res"] == "out" and self.strategy == "slide":
                belief.update_out(aim, r["corr"], r["p_ramp"])
                new_aim, _ = belief.best_aim()
                slid += 1
                s = self.slide(aim, new_aim, self.rows[r["c0"]]["tcp"][0])
                aim = new_aim
                self.drops += s in ("dropped", "latched")
                if s == "latched":
                    r = dict(res="latched")
                elif s == "dropped":
                    r = self.push(aim, x1, phase="push", decide=False)
                else:
                    belief.update_out(aim, None)
                    aim, _ = belief.best_aim()
            if r["res"] == "in_no_click":  # partial insertion: push deeper along the axis (force-limited)
                deeper += 1
                if self.push(aim, x1 + 0.004, phase="deeper", decide=False)["res"] == "latched":
                    r = dict(res="latched")
            if r["res"] == "latched":
                self.result.update(success=True)
                break
            if self.strategy == "blind":
                break
            # retract stiffly, then re-aim
            self.stiff_to([self.pre_x, *self.rows[-1]["cmd"][1:]], T=0.35, phase="retract")
            if r["res"] == "out" and self.strategy != "slide":
                if self.strategy == "spiral":
                    n_spiral += 1
                    rr = 0.001 * np.sqrt(n_spiral)
                    aim = aim0 + rr * np.array([np.cos(n_spiral * GOLDEN), np.sin(n_spiral * GOLDEN)])
                elif self.strategy == "learned":
                    aim = aim + r["corr"]
                elif self.strategy == "oracle":
                    aim = aim + r["true_corr"]
                elif self.strategy == "belief":
                    belief.update_out(aim, r["corr"], r["p_ramp"])
                    aim, _ = belief.best_aim()
            aim = np.clip(aim, -0.008, 0.008)
            self.stiff_to([self.pre_x, *aim], T=0.35, phase="reaim")
        F = np.array([row["F"] for row in self.rows])
        # after a click the tip sits on the port axis (within the clearance), so minus the measured tip
        # position (believed frame) observes the lateral perception error; the drift is added back by the prior
        u_obs = -self.rows[-1]["tcp"][1:] if self.result["success"] else None
        self.result.update(tries=tries, deeper=deeper, slides=slid, drops=self.drops, time_s=self.env.time - self.t_start,
                           peak_F=float(np.linalg.norm(F, axis=1).max()), end=outcome(self.rows, self.tc), u_obs=u_obs)
        return self.rows

    sigma_default = 0.002


def run_scene(job):
    seed, jack, sigmas, n_err, strategies, models, out_dir = job
    shard = Path(out_dir) / f"scene_{seed}.json"
    if shard.exists():
        return json.loads(shard.read_text())
    env = make_env(JACKS[jack])
    got = prepare(env, seed)
    out = []
    if got is not None:
        _ex, snap = got
        port = int(env.scenario.task["cables"][0]["port"])
        pb = Probe(env)
        rng = np.random.default_rng([seed, 23])
        for sigma in sigmas:
            for e in range(n_err):
                err = np.array([rng.uniform(-0.002, 0.002), *rng.normal(0, sigma, 2)])
                for s in strategies:
                    g = Guided(env, pb, snap, port, err=err, strategy=s, models=models, noise_seed=seed * 1000 + e, prior_sigma=sigma)
                    g.run()
                    out.append(dict(scene=seed, jack=jack, sigma=sigma, err=err, strategy=s, unstable=bool(env.unstable),
                                    **{k: v for k, v in g.result.items() if k != "u_obs"}))
                    env.unstable = False
    tmp = shard.with_suffix(".tmp")
    tmp.write_text(json.dumps(out, default=lambda o: o.tolist() if hasattr(o, "tolist") else float(o)))
    tmp.rename(shard)
    return out


def summarize(res, strategies=STRATS, log=print):
    blocked = {r["scene"] for r in res if r["strategy"] == strategies[0] and r["end"] == "blocked"}
    log(f"{len(res)} runs; scenes {len({r['scene'] for r in res})}; scenes with a blocked attempt (excluded) {sorted(blocked)}")
    rows = []
    for sigma in sorted({r["sigma"] for r in res}):
        log(f"\nsigma = {sigma * 1e3:.1f} mm lateral per axis")
        for s in strategies:
            rr = [r for r in res if r["strategy"] == s and r["sigma"] == sigma and not r["unstable"] and r["scene"] not in blocked]
            if not rr:
                continue
            k = sum(r["success"] for r in rr)
            lo, hi = wilson(k, len(rr))
            ok = [r for r in rr if r["success"]]
            row = dict(sigma=sigma, strategy=s, n=len(rr), success=k / len(rr), lo=lo, hi=hi, time_ok=float(np.mean([r["time_s"] for r in ok])) if ok else float("nan"),
                       tries=float(np.mean([r["tries"] for r in rr])), first_try=float(np.mean([r["success"] and r["tries"] == 1 for r in rr])),
                       peak_F=float(np.median([r["peak_F"] for r in rr])))
            rows.append(row)
            log(f"  {s:8s} success {row['success']:.2f} [{lo:.2f}, {hi:.2f}] (n={len(rr)})  first-try {row['first_try']:.2f}  mean tries {row['tries']:.2f}"
                f"  mean time-to-seat {row['time_ok']:.2f} s  median peak |F| {row['peak_F']:.0f} N")
    return rows


def paired(res, a, b, sigma, n_boot=4000):
    """Paired success difference a - b over identical (scene, error) cases, bootstrap 95 % interval."""
    blocked = {r["scene"] for r in res if r["end"] == "blocked"}
    by = {}
    for r in res:
        if r["scene"] in blocked or r["unstable"] or r["sigma"] != sigma:
            continue
        by.setdefault((r["scene"], tuple(np.round(r["err"], 7))), {})[r["strategy"]] = r["success"]
    d = np.array([v[a] - v[b] for v in by.values() if a in v and b in v], float)
    if len(d) == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(0)
    bs = rng.choice(d, (n_boot, len(d))).mean(1)
    return float(d.mean()), float(np.quantile(bs, 0.025)), float(np.quantile(bs, 0.975))


def wilson(k, n, z=1.96):
    if n == 0:
        return float("nan"), float("nan")
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return c - h, c + h


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=24)
    ap.add_argument("--n_err", type=int, default=15)
    ap.add_argument("--sigmas", type=float, nargs="+", default=[0.001, 0.002])
    ap.add_argument("--strategies", nargs="+", default=list(STRATS))
    ap.add_argument("--jack", default="nominal", choices=sorted(JACKS))
    ap.add_argument("--workers", type=int, default=12)
    ap.add_argument("--name", default="e2")
    ap.add_argument("--out", default=None, help="shard directory (default runs/insertion_state/<name>)")
    ap.add_argument("--train", default="e1", help="E1 run name or shard directory")
    ap.add_argument("--seed0", type=int, default=3000)
    args = ap.parse_args()
    models = train_models(args.train)
    out_dir = Path(args.out) if args.out else OUT / args.name
    out_dir.mkdir(parents=True, exist_ok=True)
    jobs = [(s, args.jack, args.sigmas, args.n_err, args.strategies, models, str(out_dir)) for s in range(args.seed0, args.seed0 + args.n)]
    res = []
    with Pool(args.workers) as p:
        for i, chunk in enumerate(p.imap_unordered(run_scene, jobs), 1):
            res += chunk
            print(f"  scene {i}/{len(jobs)} done", flush=True)
    summarize(res, args.strategies)
    for sigma in args.sigmas:
        for a, b in (("belief", "spiral"), ("slide", "belief"), ("belief", "learned")):
            if a in args.strategies and b in args.strategies:
                d, lo, hi = paired(res, a, b, sigma)
                print(f"  sigma {sigma * 1e3:.0f} mm  {a} - {b}: {d:+.3f} [{lo:+.3f}, {hi:+.3f}]")


if __name__ == "__main__":
    main()
''')

In [ ]:
write_module('belief.py', r'''"""Grid belief over the unknown lateral offset of the hole, updated from failed contacts.

Coordinates are lateral (y, z) in the believed port frame. Aiming the tip at `a` puts it at
m = a + u relative to the true port at first contact, where u (perception error + controller drift) is
unknown but constant within an episode. A failed contact contributes two pieces of evidence:
  - negative information: the plug did not enter at m, likelihood 1 - p_in(m) from the E1 capture map;
  - the contact signals: the regressor's correction c_hat ~ N(-m, S_class), mixed over ramp / face by the
    classifier's probability.
The next aim maximises the posterior probability of entering, sum_u post(u) p_in(a + u)."""
from __future__ import annotations

import numpy as np


class CaptureMap:
    """P(entry | tip offset at contact), a smoothed 2-D histogram on a regular grid."""

    def __init__(self, m, entered, half=0.012, res=0.0005, smooth=1.0):
        n = int(round(2 * half / res)) + 1
        self.half, self.res, self.n = half, res, n
        idx = np.clip(np.round((np.asarray(m) + half) / res).astype(int), 0, n - 1)
        hit = np.zeros((n, n))
        tot = np.zeros((n, n))
        np.add.at(tot, (idx[:, 0], idx[:, 1]), 1.0)
        np.add.at(hit, (idx[:, 0], idx[:, 1]), np.asarray(entered, float))
        if smooth > 0:
            from scipy.ndimage import gaussian_filter

            hit, tot = gaussian_filter(hit, smooth), gaussian_filter(tot, smooth)
        # pseudo-count towards 0 far from data (outside the sampled disk the plug never enters)
        self.P = hit / (tot + 0.05)

    def __call__(self, m):
        idx = np.round((np.asarray(m) + self.half) / self.res).astype(int)
        ok = (idx >= 0).all(-1) & (idx < self.n).all(-1)
        out = np.zeros(idx.shape[:-1])
        out[ok] = self.P[idx[ok][:, 0], idx[ok][:, 1]]
        return out


class HoleBelief:
    def __init__(self, cap: CaptureMap, sigma, S_ramp, S_face, mean=(0.0, 0.0), half=0.010, res=0.00025, drift_sigma=0.0003):
        g = np.arange(-half, half + 1e-12, res)
        U = np.stack(np.meshgrid(g, g, indexing="ij"), -1)  # (n, n, 2)
        self.U = U.reshape(-1, 2)
        self.n = len(g)
        self.cap = cap
        d = self.U - np.asarray(mean)
        sig2 = np.broadcast_to(np.asarray(sigma, float) ** 2, (2,))
        self.logp = -0.5 * np.sum(d * d / sig2, -1)
        self.S = {"ramp": np.asarray(S_ramp), "face": np.asarray(S_face)}
        self.Si = {k: np.linalg.inv(v) for k, v in self.S.items()}
        self.logdet = {k: np.linalg.slogdet(v)[1] for k, v in self.S.items()}
        self.drift_sigma, self.res = drift_sigma, res
        self._normalise()

    def _normalise(self):
        self.logp -= self.logp.max()
        p = np.exp(self.logp)
        self.p = p / p.sum()

    def _diffuse(self):
        """Process noise between tries (the plug can shift in the grasp after a failed push)."""
        if self.drift_sigma <= 0:
            return
        from scipy.ndimage import gaussian_filter

        P = gaussian_filter(self.p.reshape(self.n, self.n), self.drift_sigma / self.res)
        self.logp = np.log(P.reshape(-1) + 1e-300)
        self._normalise()

    def update_out(self, aim, c_hat=None, p_ramp=0.5):
        m = np.asarray(aim)[None] + self.U
        ll = np.log(np.clip(1.0 - self.cap(m), 1e-3, 1.0))
        if c_hat is not None:
            r = np.asarray(c_hat)[None] + m  # c_hat - (-m)
            comp = []
            for k, w in (("ramp", p_ramp), ("face", 1.0 - p_ramp)):
                q = np.einsum("ni,ij,nj->n", r, self.Si[k], r)
                comp.append(np.log(max(w, 1e-6)) - 0.5 * (q + self.logdet[k]))
            ll = ll + np.logaddexp(*comp)
        self.logp = np.log(self.p + 1e-300) + ll
        self._normalise()
        self._diffuse()

    def mean(self):
        return self.p @ self.U

    def best_aim(self, half=0.008, res=0.0005, mass=0.999):
        """argmax_a sum_u post(u) p_in(a + u), with u restricted to the support holding `mass` of the belief."""
        order = np.argsort(self.p)[::-1]
        k = int(np.searchsorted(np.cumsum(self.p[order]), mass)) + 1
        sup, w = self.U[order[:k]], self.p[order[:k]]
        g = np.arange(-half, half + 1e-12, res)
        A = np.stack(np.meshgrid(g, g, indexing="ij"), -1).reshape(-1, 2)
        best, best_s = np.zeros(2), -1.0
        for chunk in np.array_split(A, max(1, len(A) * len(sup) // 2_000_000 + 1)):
            s = (self.cap(chunk[:, None] + sup[None]) * w[None]).sum(1)
            i = int(np.argmax(s))
            if s[i] > best_s:
                best, best_s = chunk[i], float(s[i])
        return best, best_s
''')

In [ ]:
write_module('seq_model.py', r'''"""Temporal contact-state model: a small 1-D CNN ensemble over the high-rate wrench window around the
detected contact (plus the 20 Hz kinematic features), predicting the contact class and a Gaussian over the
lateral correction. Compared against the linear models of analyze_e1 at the same latency, leave-scenes-out.

Window: control steps c0-2 .. c0+k, each contributing its high-rate wrench samples (ts["Fh"], 100 Hz) and
the per-step features (lag xyz, axial travel, joint torques) repeated across its samples."""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn

import analyze_e1 as A

PRE = 2


def window(a, i, k, rng=None):
    """(channels, length) window for attempt a, plus the detection step; None if no contact is detected."""
    ts = a["ts"]
    rng = rng or np.random.default_rng([i, 5])
    F, M, tau = A.noisy(a, i)
    c0 = A.first_contact(F, ts["phase"])
    if c0 is None:
        return None, None
    hr = ts.get("Fh")
    n = len(ts["t"])
    steps = [min(max(s, 0), n - 1) for s in range(c0 - PRE, c0 + k + 1)]
    b = max(0, c0 - 1)
    lag = ts["tcp"] - ts["cmd"]
    cols = []
    for s in steps:
        low = np.concatenate([lag[s] - lag[b], [ts["tcp"][s, 0] - ts["tcp"][b, 0]], tau[s] - tau[b]]) * np.r_[[1e3] * 4, [1.0] * 7]
        if hr is not None:
            w = hr[s] + rng.normal(0, 1, hr[s].shape) * np.r_[[A.SIG_F] * 3, [A.SIG_M] * 3]
        else:
            w = np.concatenate([F[s], M[s]])[None]
        for r in w:
            cols.append(np.concatenate([r[:3], r[3:] * 10.0, low]))
    return np.array(cols, np.float32).T, c0


def build(res, k):
    X, y, corr, scene = [], [], [], []
    for i, a in enumerate(res):
        x, c0 = window(a, i, k)
        if x is None:
            continue
        X.append(x)
        y.append(A.CLASSES.index(A.label(a["ts"], c0)))
        corr.append(-a["ts"]["tip_true"][c0, 1:] * 1e3)  # mm
        scene.append(a["scene"])
    return np.stack(X), np.array(y), np.array(corr, np.float32), np.array(scene)


class Net(nn.Module):
    def __init__(self, c_in, width=64, n_cls=4):
        super().__init__()
        self.f = nn.Sequential(nn.Conv1d(c_in, width, 3, padding=1), nn.GELU(), nn.Conv1d(width, width, 3, padding=1), nn.GELU(),
                               nn.Conv1d(width, width, 3, padding=1), nn.GELU())
        self.head = nn.Sequential(nn.Linear(2 * width + c_in, width), nn.GELU(), nn.Linear(width, n_cls + 4))
        self.n_cls = n_cls

    def forward(self, x):
        h = self.f(x)
        z = torch.cat([h.mean(-1), h.amax(-1), x[:, :, -1]], -1)
        o = self.head(z)
        return o[:, : self.n_cls], o[:, self.n_cls : self.n_cls + 2], o[:, self.n_cls + 2 :].clamp(-6, 6)


def train_ensemble(X, y, corr, n_members=5, steps=3000, bs=512, lr=2e-3, device="cpu", seed=0, log=print):
    mu = X.mean((0, 2), keepdims=True)
    sd = X.std((0, 2), keepdims=True) + 1e-6
    Xt = torch.tensor((X - mu) / sd, device=device)
    yt = torch.tensor(y, device=device)
    ct = torch.tensor(corr, device=device)
    out_mask = torch.tensor(np.isin(y, [1, 2]), device=device)
    w_cls = torch.tensor(1.0 / np.maximum(1, np.bincount(y, minlength=4)), dtype=torch.float32, device=device)
    w_cls = w_cls / w_cls.mean()
    members = []
    for mi in range(n_members):
        torch.manual_seed(seed * 100 + mi)
        net = Net(X.shape[1]).to(device)
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
        for it in range(steps):
            idx = torch.randint(0, len(Xt), (bs,), device=device)
            logits, m, lv = net(Xt[idx])
            loss = nn.functional.cross_entropy(logits, yt[idx], weight=w_cls)
            om = out_mask[idx]
            if om.any():
                r = ct[idx][om] - m[om]
                loss = loss + 0.5 * (r * r * torch.exp(-lv[om]) + lv[om]).sum(-1).mean() * 0.2
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
        members.append(net.eval())
        log(f"  member {mi}: final loss {loss.item():.3f}")
    return dict(members=members, mu=mu, sd=sd, device=device)


@torch.no_grad()
def predict(model, X):
    """Class probabilities and a Gaussian over the correction [mm] (ensemble mixture moments)."""
    Xt = torch.tensor((X - model["mu"]) / model["sd"], device=model["device"])
    P, Ms, Vs = [], [], []
    for net in model["members"]:
        logits, m, lv = net(Xt)
        P.append(torch.softmax(logits, -1))
        Ms.append(m)
        Vs.append(torch.exp(lv))
    P = torch.stack(P).mean(0)
    Ms, Vs = torch.stack(Ms), torch.stack(Vs)
    mean = Ms.mean(0)
    var = Vs.mean(0) + Ms.var(0, unbiased=False)
    return P.cpu().numpy(), mean.cpu().numpy(), var.cpu().numpy()


def evaluate(res, ks=(0, 1, 2), device="cpu", steps=3000, n_members=5, log=print):
    """Leave-scenes-out comparison of the CNN ensemble and the linear models at each latency k."""
    rows = []
    for k in ks:
        X, y, corr, scene = build(res, k)
        Xl, yl, corrl, scenel, _ = A.build(res, k)
        assert len(y) == len(yl)
        te = A.folds(scene, n_folds=5)[0]
        model = train_ensemble(X[~te], y[~te], corr[~te], n_members=n_members, device=device, steps=steps, log=log)
        P, mean, var = predict(model, X[te])
        pred = P.argmax(1)
        out = np.isin(y[te], [1, 2])
        z2 = ((corr[te] - mean) ** 2 / var).sum(1)
        # linear baseline on the same split
        sd = A.Std().fit(Xl[~te])
        Wb = A.fit_logreg(sd(Xl[~te]), (yl[~te] != 0).astype(int), 2)
        W4 = A.fit_logreg(sd(Xl[~te]), yl[~te], 4)
        m_tr = ~te & np.isin(yl, [1, 2])
        B = A.fit_ridge(sd(Xl[m_tr]), corrl[m_tr], lam=10.0)
        pl = A.predict_ridge(B, sd(Xl[te])) * 1e3
        row = dict(
            k=k, ms=50 * (k + 1), n_test=int(te.sum()),
            cnn_inout=A.balanced_acc((y[te] != 0).astype(int), (pred != 0).astype(int), 2),
            lin_inout=A.balanced_acc((yl[te] != 0).astype(int), A.predict_logreg(Wb, sd(Xl[te])), 2),
            cnn_4cls=A.balanced_acc(y[te], pred, 4), lin_4cls=A.balanced_acc(yl[te], A.predict_logreg(W4, sd(Xl[te])), 4),
            cnn_dir_ramp=float(np.median(A.ang_err(mean[out & (y[te] == 1)], corr[te][out & (y[te] == 1)]))),
            lin_dir_ramp=float(np.median(A.ang_err(pl[out & (y[te] == 1)], corr[te][out & (y[te] == 1)]))),
            cnn_dir_face=float(np.median(A.ang_err(mean[out & (y[te] == 2)], corr[te][out & (y[te] == 2)]))),
            lin_dir_face=float(np.median(A.ang_err(pl[out & (y[te] == 2)], corr[te][out & (y[te] == 2)]))),
            cnn_cover_90=float(np.mean(z2[out] < 4.605)),  # chi2(2) 90 % ellipse coverage on out contacts
        )
        log(row)
        rows.append(row)
    return rows
''')

In [ ]:
write_module('e3_recurring.py', r'''"""E3: remembering the port. Part of the perception error recurs: a bias b_k per panel / camera context k,
plus per-episode noise. A stream of episodes visits K contexts with sticky dwell times; a visual cue shows
the true context with probability rho. Every method runs the belief controller; they differ only in the
prior over the lateral offset at the start of an episode:
  none     N(0, sigma_b^2 + sigma_e^2)                      (memoryless)
  pooled   one memory over all episodes                      (ignores context)
  cue      one memory per cued context                       (trusts the cue)
  oracle   one memory per true context                       (upper bound)
A memory is a Gaussian over the recurring part of the lateral perception error: after a click the tip sits
on the port axis, so minus the measured tip position (believed frame) observes the error of that episode
(no privileged information). The controller adds its mean drift on top of the remembered error."""
from __future__ import annotations

import argparse
import json
import sys
from multiprocessing import Pool
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).parent))
from common import JACKS, Probe, make_env, prepare  # noqa: E402
from e2_guided import OUT, Guided, train_models  # noqa: E402

METHODS = ("none", "pooled", "cue", "oracle")
SIG_Q = 0.0006  # where in the clearance the plug seats + drift variation between episodes (assumed)


class Memory:
    def __init__(self, sigma_b, sigma_e):
        self.sb2, self.se2 = sigma_b**2, sigma_e**2 + SIG_Q**2
        self.obs = {}

    def prior(self, key):
        """Prior over u (mean, per-axis sigma) for an episode whose memory key is `key`."""
        o = self.obs.get(key, [])
        n = len(o)
        if n == 0:
            return np.zeros(2), np.sqrt(self.sb2 + self.se2)
        post_var = 1.0 / (1.0 / self.sb2 + n / self.se2)
        post_mean = post_var * (np.sum(o, 0) / self.se2)
        return post_mean, np.sqrt(post_var + self.se2)

    def add(self, key, u_obs):
        self.obs.setdefault(key, []).append(np.asarray(u_obs))


def make_stream(seed, n, k_ctx, rho, dwell, sigma_b):
    rng = np.random.default_rng([seed, 31])
    biases = rng.normal(0, sigma_b, (k_ctx, 2))
    ctx, cue = [], []
    c = int(rng.integers(k_ctx))
    for _ in range(n):
        if rng.random() > 1 - 1 / dwell:
            c = int(rng.integers(k_ctx))
        ctx.append(c)
        cue.append(c if rng.random() < rho else int(rng.integers(k_ctx)))
    return biases, ctx, cue


def run_stream(job):
    seed, scenes, a = job["seed"], job["scenes"], job["args"]
    shard = Path(a["out_dir"]) / f"stream_{seed}.json"
    if shard.exists():
        return json.loads(shard.read_text())
    env = make_env(JACKS[a["jack"]])
    pb = Probe(env)
    models = job["models"]
    biases, ctx, cue = make_stream(seed, len(scenes), a["k_ctx"], a["rho"], a["dwell"], a["sigma_b"])
    rng = np.random.default_rng([seed, 37])
    mem = {m: Memory(a["sigma_b"], a["sigma_e"]) for m in METHODS}
    out = []
    for e, scene in enumerate(scenes):
        got = prepare(env, scene)
        if got is None:
            continue
        _ex, snap = got
        port = int(env.scenario.task["cables"][0]["port"])
        err = np.array([rng.uniform(-0.002, 0.002), *(biases[ctx[e]] + rng.normal(0, a["sigma_e"], 2))])
        for m in METHODS:
            key = {"none": None, "pooled": 0, "cue": cue[e], "oracle": ctx[e]}[m]
            mean, sig = mem[m].prior(key) if key is not None else (np.zeros(2), np.sqrt(a["sigma_b"] ** 2 + a["sigma_e"] ** 2))
            g = Guided(env, pb, snap, port, err=err, strategy="belief", models=models, prior_mean=mean, prior_sigma=sig, noise_seed=seed * 10000 + e)
            g.run()
            r = g.result
            if r["success"] and key is not None and r.get("u_obs") is not None:
                mem[m].add(key, r["u_obs"])
            first_visit = ctx[e] not in ctx[:e]
            out.append(dict(stream=seed, episode=e, scene=scene, method=m, ctx=ctx[e], cue=cue[e], first_visit=first_visit,
                            unstable=bool(env.unstable), **{k: v for k, v in r.items() if k != "u_obs"}))
            env.unstable = False
    tmp = shard.with_suffix(".tmp")
    tmp.write_text(json.dumps(out, default=lambda o: o.tolist() if hasattr(o, "tolist") else float(o)))
    tmp.rename(shard)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--streams", type=int, default=12)
    ap.add_argument("--episodes", type=int, default=40)
    ap.add_argument("--k_ctx", type=int, default=3)
    ap.add_argument("--rho", type=float, default=0.8)
    ap.add_argument("--dwell", type=float, default=5.0)
    ap.add_argument("--sigma_b", type=float, default=0.002)
    ap.add_argument("--sigma_e", type=float, default=0.0005)
    ap.add_argument("--scene_pool", type=int, nargs=2, default=[5000, 5400])
    ap.add_argument("--train", default="e1", help="E1 run name or shard directory")
    ap.add_argument("--jack", default="nominal", choices=sorted(JACKS))
    ap.add_argument("--workers", type=int, default=12)
    ap.add_argument("--name", default="e3")
    ap.add_argument("--out", default=None, help="shard directory (default runs/insertion_state/<name>)")
    args = ap.parse_args()
    out_dir = Path(args.out) if args.out else OUT / args.name
    out_dir.mkdir(parents=True, exist_ok=True)
    a = vars(args) | {"out_dir": str(out_dir)}
    models = train_models(args.train)
    pool = np.arange(*args.scene_pool)
    jobs = []
    for s in range(args.streams):
        rng = np.random.default_rng([s, 41])
        jobs.append(dict(seed=s, scenes=[int(x) for x in rng.choice(pool, args.episodes, replace=False)], args=a, models=models))
    res = []
    with Pool(args.workers) as p:
        for i, chunk in enumerate(p.imap_unordered(run_stream, jobs), 1):
            res += chunk
            print(f"  stream {i}/{len(jobs)} done", flush=True)
    summarize(res)


def summarize(res, log=print):
    """Episodes where any method was blocked by a neighbouring cable are dropped for all methods."""
    bad = {(r["stream"], r["episode"]) for r in res if r["unstable"] or r["end"] == "blocked"}
    res = [r for r in res if (r["stream"], r["episode"]) not in bad]
    log(f"{len(res)} episode-method runs ({len(bad)} blocked / unstable episodes excluded)")
    rows = []
    for m in METHODS:
        rr = [r for r in res if r["method"] == m]
        if not rr:
            continue
        ret = [r for r in rr if not r["first_visit"]]
        row = dict(method=m, n=len(rr), success=float(np.mean([r["success"] for r in rr])),
                   first_try=float(np.mean([r["success"] and r["tries"] == 1 for r in rr])),
                   first_try_returns=float(np.mean([r["success"] and r["tries"] == 1 for r in ret])) if ret else float("nan"),
                   tries=float(np.mean([r["tries"] for r in rr])), time_s=float(np.mean([r["time_s"] for r in rr])))
        rows.append(row)
        log(f"  {m:7s} success {row['success']:.2f}  first-try {row['first_try']:.2f}  first-try on returns {row['first_try_returns']:.2f}"
            f"  mean tries {row['tries']:.2f}  mean time {row['time_s']:.2f} s")
    return rows


if __name__ == "__main__":
    main()
''')

In [ ]:
import json
import time

import numpy as np
import torch
from IPython.display import Image, display

from robogym.tools import run_live

PRESET = os.environ.get("INSERTION_PRESET", "a100" if torch.cuda.is_available() else "smoke")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORKERS = os.cpu_count()
P = dict(
    smoke=dict(jacks=["nominal"], e1_scenes=4, e1_radii=[1.0, 2.5, 4.0], cnn_steps=200, cnn_members=2,
               e2_scenes=3, e2_err=2, e2_sigmas=[0.002], e3_streams=2, e3_eps=6),
    a100=dict(jacks=["nominal", "tight"], e1_scenes=200, e1_radii=None, cnn_steps=4000, cnn_members=5,
              e2_scenes=48, e2_err=12, e2_sigmas=[0.001, 0.002, 0.003], e3_streams=24, e3_eps=40),
)[PRESET]
RUN = pathlib.Path(ROOT) / "runs" / "insertion_state_nb" / PRESET
RUN.mkdir(parents=True, exist_ok=True)
SUMMARY = RUN / "summary.json"
summary = json.loads(SUMMARY.read_text()) if SUMMARY.exists() else {}


def save_summary():
    SUMMARY.write_text(json.dumps(summary, indent=1, default=float))


def run(script, *args):
    run_live([sys.executable, str(EXP / script), *map(str, args)], cwd=str(ROOT))


print(PRESET, DEVICE, "workers", WORKERS, "->", RUN)
print(P)

## 0. Check: the scene builds, the expert reaches the insertion, timing
The VM runs the committed RoboGym; the laptop numbers came from a working tree with unrelated edits
(cube stacking, hand-over), so this checks the cable task behaves the same.

In [ ]:
from common import JACKS, Attempt, Probe, make_env, prepare  # noqa: E402

for jack in P["jacks"]:
    env = make_env(JACKS[jack])
    t0 = time.time()
    ex, snap = prepare(env, 1000)
    t_prep = time.time() - t0
    pb = Probe(env)
    port = int(env.scenario.task["cables"][0]["port"])
    t0 = time.time()
    rows = Attempt(env, pb, snap, port).run()
    print(f"{jack}: prepare {t_prep:.1f}s, one attempt {time.time() - t0:.1f}s, latched {rows[-1]['latched'] >= 0}, "
          f"high-rate wrench per step {rows[-1]['Fh'].shape}")

## 1. E1: blind insertions with controlled port-estimate errors
From the pre-insertion snapshot of each scene: lateral error on a polar grid (0-8 mm x 12 angles),
axial error U(-2, 2) mm, one compliant force-limited push. Every step logs the 20 Hz signals, the 100 Hz
wrench and the privileged labels. One shard per scene.

In [ ]:
E1 = {}
for jack in P["jacks"]:
    E1[jack] = RUN / f"e1_{jack}"
    args = ["--n", P["e1_scenes"], "--seed0", 1000, "--jack", jack, "--workers", WORKERS, "--out", E1[jack]]
    if P["e1_radii"]:
        args += ["--radii", json.dumps(P["e1_radii"])]
    t0 = time.time()
    run("e1_offset_sweep.py", *args)
    print(f"{jack}: {time.time() - t0:.0f}s")

## 2. Identifiability with linear models (leave-scenes-out)
Contact is detected from the noisy tared wrench (|F| > 2.5 N); labels at that moment: `in` (enters the
channel), `ramp` (on the chamfer), `face` (beyond it), `blocked` (gripper or plug on a neighbouring cable).
In/out and 4-class logistic models, ridge regression of the lateral correction, 50-200 ms after contact.

In [ ]:
import analyze_e1 as A  # noqa: E402

for jack in P["jacks"]:
    print(f"\n========== {jack} jack")
    res = A.load_runs(E1[jack])
    rows = A.analyze(res, fig_dir=RUN / f"figs_{jack}")
    summary.setdefault(jack, {})["linear"] = rows
    display(Image(str(RUN / f"figs_{jack}" / "e1_contact_state.png")))
    del res
save_summary()

## 3. Temporal CNN ensemble on the 100 Hz window vs the linear models (GPU)
Same split (one fifth of the scenes held out), same latency. The CNN also predicts a Gaussian over the
correction; `cnn_cover_90` is the coverage of its 90 % ellipse on out-of-jack contacts (calibration matters
for the belief search).

In [ ]:
import seq_model as S  # noqa: E402

for jack in P["jacks"]:
    print(f"\n========== {jack} jack")
    res = A.load_runs(E1[jack])
    rows = S.evaluate(res, ks=(0, 1, 2), device=DEVICE, steps=P["cnn_steps"], n_members=P["cnn_members"])
    summary[jack]["cnn"] = rows
    del res
save_summary()
for jack in P["jacks"]:
    print(jack)
    for r in summary[jack]["cnn"]:
        print(f"  {r['ms']:3d} ms  in/out cnn {r['cnn_inout']:.3f} lin {r['lin_inout']:.3f} | 4-class cnn {r['cnn_4cls']:.3f} lin {r['lin_4cls']:.3f}"
              f" | dir ramp cnn {r['cnn_dir_ramp']:.0f} lin {r['lin_dir_ramp']:.0f} deg | dir face cnn {r['cnn_dir_face']:.0f} lin {r['lin_dir_face']:.0f} deg"
              f" | cover90 {r['cnn_cover_90']:.2f}")

## 4. E2: closed-loop insertion with retries (new scenes)
Shared in/out detector and push-deeper rule; all strategies start from the same drift-compensated aim.
`blind` one push | `spiral` sunflower search | `learned` jump by the predicted correction | `belief` grid
belief with negative information, aims where entry is most likely | `slide` belief, but slides to the new
aim under a light preload instead of retracting | `oracle` jumps by the true miss. Budget 9 s, 6 tries.
Scenes where the gripper hits a neighbouring cable are reported separately (lateral search cannot fix them).

In [ ]:
import e2_guided as E2mod  # noqa: E402

for jack in P["jacks"]:
    print(f"\n========== {jack} jack")
    out = RUN / f"e2_{jack}"
    t0 = time.time()
    run("e2_guided.py", "--n", P["e2_scenes"], "--seed0", 3000, "--n_err", P["e2_err"], "--sigmas", *P["e2_sigmas"],
        "--jack", jack, "--workers", WORKERS, "--train", E1[jack], "--out", out)
    print(f"{jack}: {time.time() - t0:.0f}s")
    res = [r for f in sorted(out.glob("scene_*.json")) for r in json.loads(f.read_text())]
    summary[jack]["e2"] = E2mod.summarize(res, log=lambda *a: None)
    summary[jack]["e2_paired"] = {f"{a}-{b}@{s}": E2mod.paired(res, a, b, s) for s in P["e2_sigmas"]
                                  for a, b in (("belief", "spiral"), ("belief", "learned"), ("slide", "belief"), ("oracle", "belief"))}
save_summary()

## 5. E3: remembering the port
K=3 panel/camera contexts, each with a recurring lateral bias (sigma_b = 2 mm) plus per-episode noise
(0.5 mm); sticky switching (mean dwell 5 episodes), cue correct with probability 0.8. All methods run the
belief controller and differ only in the prior: none | pooled memory | memory per cued context | per true
context. The memory sees only the robot's own measurement at the click.

In [ ]:
import e3_recurring as E3mod  # noqa: E402

for jack in P["jacks"]:
    print(f"\n========== {jack} jack")
    out = RUN / f"e3_{jack}"
    t0 = time.time()
    run("e3_recurring.py", "--streams", P["e3_streams"], "--episodes", P["e3_eps"], "--jack", jack, "--workers", WORKERS,
        "--train", E1[jack], "--out", out)
    print(f"{jack}: {time.time() - t0:.0f}s")
    res = [r for f in sorted(out.glob("stream_*.json")) for r in json.loads(f.read_text())]
    summary[jack]["e3"] = E3mod.summarize(res)
save_summary()

## 6. Summary

In [ ]:
for jack in P["jacks"]:
    s = summary.get(jack, {})
    print(f"\n========== {jack} jack")
    for r in s.get("linear", []):
        if r["k"] == 1:
            print(f"linear @100 ms: in/out {r['inout_logreg']:.3f}, 4-class {r['cls4_logreg']:.3f}, dir ramp {r.get('dir_ramp_ridge', float('nan')):.0f} deg")
    for r in s.get("cnn", []):
        if r["k"] == 1:
            print(f"cnn    @100 ms: in/out {r['cnn_inout']:.3f}, 4-class {r['cnn_4cls']:.3f}, dir ramp {r['cnn_dir_ramp']:.0f} deg, cover90 {r['cnn_cover_90']:.2f}")
    print("E2 success (excl. blocked scenes):")
    for sig in P["e2_sigmas"]:
        cells = [r for r in s.get("e2", []) if r["sigma"] == sig]
        print(f"  sigma {sig * 1e3:.0f} mm: " + "  ".join(f"{r['strategy']} {r['success']:.2f}" for r in cells))
    for k, (d, lo, hi) in s.get("e2_paired", {}).items():
        print(f"  {k}: {d:+.3f} [{lo:+.3f}, {hi:+.3f}]")
    print("E3:")
    for r in s.get("e3", []):
        print(f"  {r['method']:7s} success {r['success']:.2f}, first-try {r['first_try']:.2f}, first-try on returns {r['first_try_returns']:.2f}, tries {r['tries']:.2f}")
print(f"\nsummary -> {SUMMARY}")

The full summary as JSON, so the results travel back with the notebook outputs even when Drive is not
mounted on the VM (VS Code keeps cell outputs in the local .ipynb).

In [ ]:
print(json.dumps(summary, default=float))